# Sera-Z — Latih Model 1 (YOLOv11n-cls) dari dataset Hugging Face di Colab

Notebook ini menjalankan pipeline penuh:

1. Unduh dataset `ashraq/fashion-product-images-small` dari Hugging Face.
2. Filter label dan gender, lalu split 70/15/15 stratified per kelas.
3. Simpan ke struktur folder Ultralytics (`train/val/test/<kelas>/`).
4. Latih YOLOv11n-cls dengan GPU Colab.
5. Evaluasi pada test set.
6. Ekspor TFLite int8 dan unduh hasilnya.

**Kelas model: 10** (9 jenis pakaian + `other`). Kelas `other` berisi foto
objek bukan pakaian, supaya aplikasi bisa bilang "ini bukan pakaian" saat
kamera diarahkan ke laptop atau meja.

**Sebelum mulai:** runtime harus GPU. Menu `Runtime > Change runtime type > T4 GPU`.

**Catatan jujur:** dataset HF ini adalah port dari Kaggle
`paramaggarwal/fashion-product-images-small`, yaitu sumber yang sudah dipakai
pipeline lama (66,9 persen data latih). Jadi hasilnya kemungkinan besar setara,
bukan lebih tinggi. Gunakan notebook ini untuk melatih ulang bersih di Colab
tanpa Kaggle API key. Peningkatan kualitas yang nyata datang dari foto asli
pengguna, bukan dari menambah volume dataset katalog yang sama.

In [ ]:
# 1. Cek GPU dan versi library
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Ambil repo proyek
# Ganti REPO_URL dengan URL repo kelompok kalian, atau upload ZIP repo lalu
# ekstrak ke /content/PBL-Mobile-Kelompok2-Semester5.
REPO_URL = "https://github.com/USERNAME/PBL-Mobile-Kelompok2-Semester5.git"
BRANCH = "dev"

import os

if REPO_URL.startswith("https://github.com/USERNAME"):
    raise SystemExit("Ganti REPO_URL dulu, atau upload ZIP repo secara manual.")

if not os.path.isdir("/content/PBL-Mobile-Kelompok2-Semester5"):
    !git clone --branch {BRANCH} {REPO_URL} /content/PBL-Mobile-Kelompok2-Semester5

%cd /content/PBL-Mobile-Kelompok2-Semester5
!git pull
!git log -1 --oneline

In [ ]:
# 3. Pasang dependensi
# torch sudah ada di Colab. Ultralytics + datasets dipasang di atasnya.
!pip install -q ultralytics datasets pillow pandas scikit-learn matplotlib tqdm

import ultralytics, datasets
print("ultralytics:", ultralytics.__version__)
print("datasets:", datasets.__version__)

In [ ]:
# 4. Uji cepat dengan sebagian data (opsional, 5-10 menit)
# Jalankan sel ini untuk memastikan pipeline tidak error sebelum run penuh.
# Lewati ke sel 5 untuk langsung memakai seluruh dataset.
!python ml/scripts/09_prepare_hf.py --force --limit 3000 --out ml/data/processed/model1_smoke

!python ml/scripts/04_train_yolo11n.py --data ml/data/processed/model1_smoke \
    --epochs 3 --batch 32 --imgsz 224 --workers 2 --device 0 --name smoke_hf

In [ ]:
# 5. Siapkan dataset penuh (unduh ~270 MB, simpan ~1-2 GB, 10-20 menit)
!python ml/scripts/09_prepare_hf.py --force --out ml/data/processed/model1

### 5b. Tambahkan foto objek bukan pakaian (disarankan)

Kelas `other` membuat model bisa menolak laptop, meja, atau tangan.
Foto negatif diunduh otomatis dari **COCO 2017** lewat script
`11_download_negatives.py` (800 foto: laptop, kursi, meja, orang, botol,
TV, buku, ruangan). Kalau kamu punya foto sendiri, taruh di
`ml/data/raw/negatives/` dan script akan memakainya.

In [ ]:
# 5b. Unduh otomatis foto objek bukan pakaian, lalu siapkan dataset 10 kelas
from pathlib import Path

NEG = Path("ml/data/raw/negatives")
jumlah = len(list(NEG.rglob("*.jpg"))) if NEG.is_dir() else 0
print(f"Foto negatif ditemukan: {jumlah}")

if jumlah < 100:
    print("Mengunduh 800 foto dari COCO 2017...")
    !python ml/scripts/11_download_negatives.py --count 800

!python ml/scripts/09_prepare_hf.py --force --negatives ml/data/raw/negatives

In [ ]:
# 6. Verifikasi jumlah gambar per split
from pathlib import Path
import pandas as pd

M1 = Path("ml/data/processed/model1")
meta = pd.read_csv(M1 / "metadata.csv")
print(meta.groupby(["split", "target"]).size().unstack(fill_value=0).to_string())
print()
print("Total per split:")
print(meta.groupby("split").size().to_string())
print()
print("Kelas (urut abjad, ini urutan output model):")
print(sorted(meta["target"].unique()))

In [ ]:
# 7. Latih YOLOv11n-cls
# T4 16 GB: batch 64 aman. Kalau OOM, turunkan ke 32.
!python ml/scripts/04_train_yolo11n.py \
    --data ml/data/processed/model1 \
    --model yolo11n-cls.pt \
    --epochs 30 \
    --imgsz 224 \
    --batch 64 \
    --patience 8 \
    --workers 2 \
    --device 0 \
    --name model1_hf

In [ ]:
# 8. Evaluasi pada test set
!python ml/scripts/05_eval_model1.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf/weights/best.pt

In [ ]:
# 8b. (Disarankan) Uji pada foto asli dari HP
# Struktur: ml/data/raw/field/<kelas>/*.jpg dan ml/data/raw/field/other/*.jpg
# Nama folder kelas harus sama dengan output model, mis. tshirt, shirt, other.
!python ml/scripts/10_eval_field.py \
    --weights ml/runs/model1_hf/weights/best.pt

In [ ]:
# 9. Ekspor TFLite int8
!pip install -q tensorflow onnx onnxslim onnxruntime

!python ml/scripts/06_export_tflite.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf/weights/best.pt \
    --out-name model1_yolo11n_hf_v1.tflite

In [ ]:
# 10. Bungkus dan unduh hasil ke laptop
import shutil
from google.colab import files

shutil.make_archive("/content/model1_hf_artifacts", "zip", "/content", "PBL-Mobile-Kelompok2-Semester5/ml/runs/model1_hf")
shutil.copy(
    "/content/PBL-Mobile-Kelompok2-Semester5/assets/models/model1_yolo11n_hf_v1.tflite",
    "/content/model1_yolo11n_hf_v1.tflite",
)
for f in ("model1_hf_artifacts.zip", "model1_yolo11n_hf_v1.tflite"):
    files.download(f"/content/{f}")

In [ ]:
# 11. (Opsional) Simpan ke Google Drive agar tidak hilang saat runtime mati
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/SeraZ-ML"
!cp -r ml/runs/model1_hf "/content/drive/MyDrive/SeraZ-ML/"
!cp ml/reports/model1_test_* "/content/drive/MyDrive/SeraZ-ML/"
print("Tersimpan di /content/drive/MyDrive/SeraZ-ML")

## Setelah selesai

1. Bandingkan hasil dengan model lama: akurasi test 94,34 persen, macro F1 89,48 persen.
2. Tulis hasil baru ke `ml/reports/dataset_summary.md` dan `model1-card.md`
   (maksimal 2 angka desimal, satuan persen dengan koma).
3. Periksa urutan kelas output. Urutan mengikuti nama folder abjad:
   `dress, long_pants, long_sleeve, other, outerwear, shirt, shoes, shorts, skirt, tshirt`.
4. `assets/models/labels.txt` sudah berisi 10 kelas. Bila kamu melatih tanpa
   `other`, file itu harus disesuaikan lagi (9 kelas) agar cocok dengan model.
5. Uji juga di foto HP dengan `ml/scripts/10_eval_field.py` sebelum menilai
   hasilnya, karena akurasi test katalog tidak mewakili foto asli.

## Kalau mau kualitas lebih tinggi

Menambah dataset yang sama tidak menaikkan variasi. Yang berdampak:

- Tambah 150 foto lapangan (E10.2) ke `train/`, lalu uji ulang.
- Fine-tune dari `best.pt` hasil run ini, bukan dari ImageNet:
  `--model ml/runs/model1_hf/weights/best.pt --epochs 10`.
- Jalankan `02_balance_train.py --apply` untuk kelas `long_sleeve`, `dress`, `skirt`.
- Jangan mengubah test set agar angka tetap jujur untuk laporan.